In [ ]:
# Session 생성 - 1

from sqlalchemy import create_engine
from sqlalchemy.orm import Session

engine = create_engine("sqlite:///:memory:")

# 직접 인스턴스 생성
session = Session(engine)
print(session)
print(type(session))

session.close()
# 데이터베이스 connection을 connection pool에 반환
# commit()을 안 한 작업은 ROLLBACK
# session도 수첩이 있어서 객체를 관리하는데 close()를 호출하면 수첩에서 모두 지움

<class 'sqlalchemy.orm.session.Session'>


In [ ]:
# Session 생성 - 2

from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

engine = create_engine("sqlite:///:memory:")

# engine 등의 설정 정보가 들어 있어서 이후 호출될 때마다 동일한 설정 정보로
# 새로운 Session을 찍어내는 팩토리 객체 (sessionmaker)
SessionLocal = sessionmaker(bind=engine) # 리턴은 클래스가 아니라 인스턴스임

session1 = SessionLocal() # 공장 가동
session2 = SessionLocal() # 공장 가동

print("SessionLocal:", SessionLocal)
# SessionLocal = sessionmaker(bind=engine) 단계에서 포함되는 설정 정보

# class_='Session'          → 공장에서 찍어낼 물건의 종류를 지정; Session 클래스의 인스턴스를 만들어라
# bind=Engine(...)          → 어느 engine에 연결할지
# autoflush=True
# "아직 실제 SQL로 실행되지 않은 채" 대기하고 있는 내용을 실행시키는 것이 flush이고 확정하는게 commit
# session.query() 등으로 DB를 조회하기 직전에 아직 flush 안 된 변경사항(new/dirty/deleted)이 있으면
# 자동으로 flush()를 실행해서 조회 결과에 빠진 내용이 없게 하는 설정

# expire_on_commit=True     → commit 후 객체를 만료시킬지, 객체가 만료되면 데이터베이스에서 다시 읽어와야 함
# 특정 session을 사용하는 누군가가 session.commit()을 호출하고 나면, 그 다음 또 다른 누군가가 commit을 할 수 있음
# 따라서, commit을 하고 하면 데이터베이스에서 다시 읽어와야 할 필요가 생기는 것

print("session1:", session1)
print("session2:", session2)
print("session1 is session2:", session1 is session2) # A is B → A와 B가 동일한 객체인지 비교

session1.close()
session2.close()

SessionLocal: sessionmaker(class_='Session', bind=Engine(sqlite:///:memory:), autoflush=True, expire_on_commit=True)
session1: <sqlalchemy.orm.session.Session object at 0x00000141B1209A90>
session2: <sqlalchemy.orm.session.Session object at 0x00000141B1103B10>
session1 is session2: False


In [ ]:
# Session 생성 - 3

from sqlalchemy import create_engine, text
from sqlalchemy.orm import Session

engine = create_engine("sqlite:///:memory:")

# 컨텍스트 매니저
with Session(engine) as session:
    session.execute(text("SELECT 1"))
    conn = session.connection()
    print(conn.closed) # False

print(conn.closed) # with 블록을 벗어나면서 Session이 사용하던 Connection을 Pool에 반납

False
True


In [ ]:
# CRUD에서 사용할 테이블 생성 코드

from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, String, Boolean, DateTime
from sqlalchemy import inspect
import pprint
from sqlalchemy.orm import declarative_base, sessionmaker

Base = declarative_base()

class Todo(Base):           # Todo.__table__
    __tablename__ = "todos"
    id = Column(Integer, primary_key=True)                      # 기본키
    title = Column(String, nullable=False)                      # 필수값
    completed = Column(Boolean, default=False)                  # 기본값
    priority = Column(Integer, server_default="1")              # 기본값 (데이터베이스 서버)
    created_at = Column(DateTime, default=datetime.now)         # 기본값 (파이썬 코드; 현재 시각)

engine = create_engine("sqlite:///:memory:", echo=False)
Base.metadata.create_all(engine)

inspector = inspect(engine)

print("테이블 목록", inspector.get_table_names())
print('-'*100)

print("todos 컬럼:")
pprint.pprint(inspector.get_columns("todos"))

테이블 목록 ['todos']
----------------------------------------------------------------------------------------------------
todos 컬럼:
[{'default': None,
  'name': 'id',
  'nullable': False,
  'primary_key': 1,
  'type': INTEGER()},
 {'default': None,
  'name': 'title',
  'nullable': False,
  'primary_key': 0,
  'type': VARCHAR()},
 {'default': None,
  'name': 'completed',
  'nullable': True,
  'primary_key': 0,
  'type': BOOLEAN()},
 {'default': "'1'",
  'name': 'priority',
  'nullable': True,
  'primary_key': 0,
  'type': INTEGER()},
 {'default': None,
  'name': 'created_at',
  'nullable': True,
  'primary_key': 0,
  'type': DATETIME()}]


In [ ]:
# 객체 상태 관리 (State Management)

# Session은 지켜보는 객체 목록이 있음
# 어떤 객체 A가 그 목록에 추가되면 Session은 객체 A의 상태를 계속 지켜봄
# Session은 그 변화를 데이터베이스에 바로 반영하지 않고 내부에서 기록만 함
# Flush/Commit이 호출되면 Session은 데이터베이스에 반영/확정

# Flush는 rollback해서 취소 가능, Commit은 취소 불가능
# Flush는 변경 사항을 SQL로 만들어서 데이터베이스에 전송하는 것
# Commit은 Flush를 한 다음 논리적으로 확정하는 것

# 변경사항을 데이터베이스에 쓰는 작업에 해당하는 Create, Update, Delete는 commit() 필요
# Read는 commit() 필요하지 않음

In [ ]:
# Session이 지켜보는 객체 목록이 있음 - 1

# 특정 session에 4개의 목록이 있음
# session.new, session.dirty, session.deleted, session.identity_map

# 어떤 목록에 어떻게 추가되는가?
# ⓐ 목록 session.new: session.add(), session.add_all()
# ⓑ 목록 session.dirty": 객체의 속성을 변경
# ⓒ 목록 session.deleted: session.delete()

# 나머지 목록 session.identity_map은 현재 Session이 지켜 보고 있는 모든 객체 명단이 들어 있음

# session.identity_map 목록에 추가되는 경우는 Create/Read 두 가지
# ⓓ Create에 해당하는 session.add(객체)을 실행한 뒤 commit() 실행을 하면 session.identity_map 목록에 추가
# ⓔ   Read에 해당하는 session.query(객체).first() 등으로 조회하면 session.identity_map 목록에 추가

# ※ session.query(객체)는 query 객체를 준비하는 것이며 .first()로 실제 조회를 해야 함

# session.identity_map 목록에 추가되는 경로
# session.add_all()
#     session.add()     → session.new (즉시)
#        ↓
#  session.commit()     → session.identity_map ←     session.query().first()
#                                                    session.query().all()

# 아래 셀에서 계속

In [ ]:
# CRUD에 따른 Session의 객체 상태 변화
# 목록에 들어간다는 것은 상태를 의미

# Create (commit 전/후로 나뉨)
# session.add(), session.add_all() → session.new → commit → identity_map
# --------------------------------   -----------   ------   ------------
# 이 작업을 하면                       목록에 추가     실행     새로운 목록으로 이동


# Update (identity_map에 있어야 함)
# identity_map → 객체 속성 변경 → session.dirty, identity_map → commit → identity_map
#                               ---------------------------
#                               session.dirty에 추가되고, identity_map에도 계속 남아 있음을 의미


# Delete (identity_map에 있어야 함)
# identity_map → session.delete() → session.deleted, identity_map → commit() → 제거
#                                   -----------------------------
#                                   session.deleted에 추가되고, identity_map에도 계속 남아 있음을 의미


# Read (commit 필요 없음)
# identity_map → session.query().first(), session.query().all() → identity_map


# Create → Update
# session.add(), session.add_all() → session.new → commit → identity_map →
# identity_map → 객체 속성 변경 → session.dirty, identity_map → commit → identity_map

# Create → Delete
# session.add(), session.add_all() → session.new → commit → identity_map
# identity_map → session.delete() → session.deleted, identity_map → commit() → 제거


# 이 외에도
# session.query().all(): 조건에 맞는 모든 결과
# session.query().first(): 조건에 맞는 결과 중 첫 번째
# session.query().one(): 조건에 맞는 결과의 개수가 정확히 1개일 때만 결과 리턴
# session.get(Todo, id): Todo 테이블의 기본키가 전달한 id인 행을 리턴

In [ ]:
# Session이 지켜보는 객체 목록이 있음 - 2

from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import declarative_base, sessionmaker

Base = declarative_base()

class Todo(Base):
    __tablename__ = "todos"
    id = Column(Integer, primary_key=True)
    title = Column(String)

engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(engine)

SessionLocal = sessionmaker(bind=engine)
session = SessionLocal()

# 1. Create 단계 - add() 직후
todo = Todo(title="우유 사기")
session.add(todo) # session.add(), session.add_all() → session.new → commit → identity_map
print("add() 직후, session.new:", session.new) # session.new 목록을 출력
# add() 직후, session.new: IdentitySet([<__main__.Todo object at 0x00000141B120BE00>])
#                                       -------------------------------------------
#                                                      이 부분이 todo


session.commit()
print("commit() 후, session.new:", session.new)     # session.new에서 제거
print("commit() 후, identity_map:", dict(session.identity_map))
# commit() 후, identity_map: {(<class '__main__.Todo'>, (1,), None): <__main__.Todo object at 0x00000141B120BE00>}
#                             ---------------KEY------------------   --------------------VALUE-------------------
#                                      ----CLASS-----   -id-         --------------------todo--------------------

# 아래 셀에서 계속


add() 직후, session.new: IdentitySet([<__main__.Todo object at 0x00000141B120BE00>])
commit() 후, session.new: IdentitySet([])
commit() 후, identity_map: {(<class '__main__.Todo'>, (1,), None): <__main__.Todo object at 0x00000141B120BE00>}


In [9]:
# 2. Update 단계 - 속성 변경 직후
todo.title = "빵 사기"
print("속성 변경 후, session.dirty:", session.dirty)
session.commit()
print("Commit 후, session.dirty:", session.dirty)
print("Commit 후, session.identity_map:", dict(session.identity_map))
print('-'*100)

# 3. Delete 단계
session.delete(todo)
print("delete() 직후, session.deleted:", session.deleted)
session.commit()
print("commit() 후, session.deleted:", session.deleted)                     # 여기에도 없고
print("commit() 후, session.identity_map:", dict(session.identity_map))     # 저기에도 없음

속성 변경 후, session.dirty: IdentitySet([<__main__.Todo object at 0x00000141B120BE00>])
Commit 후, session.dirty: IdentitySet([])
Commit 후, session.identity_map: {(<class '__main__.Todo'>, (1,), None): <__main__.Todo object at 0x00000141B120BE00>}
----------------------------------------------------------------------------------------------------
delete() 직후, session.deleted: IdentitySet([<__main__.Todo object at 0x00000141B120BE00>])
commit() 후, session.deleted: IdentitySet([])
commit() 후, session.identity_map: {}


In [ ]:
# CRUD

# Create — session.add() + session.commit()
# Create — session.add_all() + session.commit()

# Update — 객체 속성 변경 + session.commit()
# Delete — session.delete() + session.commit()

In [ ]:
# Create — session.add() + session.commit()

from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, String, Boolean, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker

Base = declarative_base()

class Todo(Base):
    __tablename__ = "todos"
    id = Column(Integer, primary_key=True)
    title = Column(String, nullable=False)
    completed = Column(Boolean, default=False)
    priority = Column(Integer, server_default="1")
    created_at = Column(DateTime, default=datetime.now)

engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(engine)

SessionLocal = sessionmaker(bind=engine)
session = SessionLocal()

new_todo = Todo(title="우유 사기")  # 파이썬 객체

session.add(new_todo) # session.add(), session.add_all() → session.new → commit → identity_map
session.commit()
# commit()을 호출하면
# ⓐ. Session이 지금까지 쌓인 변경사항(session.new, session.dirty, session.deleted)을 확인
# ⓑ. Dialect가 변경사항을 실제 SQL 문(INSERT, UPDATE, DELETE 등)으로 변환
# ⓒ. Session이 engine으로부터 Connection을 얻어옴 (Pool을 통해)
# ⓓ. 해당 Connection을 통해 Dialect가 만든 SQL을 DBAPI에게 전달
# ⓔ. DBAPI가 SQL 문을 Database에 전달해서 실행 (Flush)
# ⓕ. 실행이 성공하면 COMMIT을 실행해서 트랜잭션을 확정
# ⓖ. session.new/session.dirty/session.deleted 목록은 비워지고, identity_map은 유지(또는 갱신)됨
# ⓗ. expire_on_commit=True라면 객체들이 만료 상태로 표시됨
# 확정을 한다는 점만 제외하면 flush()도 동일한 내용

# with Session(engine) as session:
#   print(session.in_transaction())   # False (아직 Connection 없음, 그냥 engine만 알고 있음)
#   session.execute(text("..."))      # 이 시점에 비로소 engine에게 Connection을 요청
#   print(session.in_transaction())   # True (이제 Connection을 얻어서 트랜잭션 시작됨)

# 확인
for todo in session.query(Todo).all():
    print(todo.id, todo.title, todo.completed, todo.priority, todo.created_at)

1 우유 사기 False 1 2026-08-22 17:22:59.833989


<img src="https://docs.sqlalchemy.org/en/20/_images/sqla_engine_arch.png" />

In [16]:
# Create — session.add_all() + session.commit()

# 위 셀에서 계속
# SessionLocal = sessionmaker(bind=engine)
session = SessionLocal()

new_todos = [
    Todo(title="책 반납하기"),
    Todo(title="운동하기", completed=True),
    Todo(title="이메일 확인", priority=5),
]
session.add_all(new_todos) # add()는 객체 하나만 추가, add_all()은 여러 객체를 리스트로 추가
session.commit()

# 확인
for todo in session.query(Todo).all():
    print(todo.id, todo.title, todo.completed, todo.priority, todo.created_at)

1 우유 사기 False 1 2026-08-22 17:22:59.833989
2 책 반납하기 False 1 2026-08-22 17:23:01.428571
3 운동하기 True 1 2026-08-22 17:23:01.429414
4 이메일 확인 False 5 2026-08-22 17:23:01.429843


In [ ]:
# Update — 조회한 객체 속성 변경 + session.commit()


todo = session.query(Todo).filter(Todo.title == "우유 사기").first()
#      --------1-----------------------2----------------------3----


# 1단계: session.query(Todo)
# Query 객체 (아직 실행 안 됨)

# 2단계: session.query(Todo).filter(Todo.title == "우유 사기")
# Query 객체 (아직 실행 안 됨)
# Query 객체에 "title이 '우유 사기'인 것만"이라는 조건(WHERE)을 추가한 상태

# 3단계: session.query(Todo).filter(Todo.title == "우유 사기").first()
# SQL이 데이터베이스에 전송되고 첫 번째 결과만 반환

print("변경 전:", todo.title, todo.completed)

todo.completed = True     # 조회해온 객체의 속성을 직접 변경
session.commit()          # 이 시점에 UPDATE SQL이 실행+확정

print("변경 후:", todo.title, todo.completed)

변경 전: 우유 사기 False
변경 후: 우유 사기 True


In [18]:
# Delete — session.delete() + session.commit()

todo = session.query(Todo).filter(Todo.title == "우유 사기").first()
# 삭제할 대상을 먼저 검색
# session.query(Todo).filter(Todo.title == "우유 사기").first()의 리턴은 Todo 인스턴스

print("삭제 전:", session.query(Todo).all()) # 모두 4개 추가된 상태임
# Todo 클래스의 __tablename__ = "todos"에 의해서 Todo 클래스는 todos 테이블과 연결
# 따라서 session.query(Todo).all()는 todos 테이블의 모든 행을 조회해서 Todo 객체로 변환한 리스트를 리턴
# 4개의 Todo 인스턴스 리스트

session.delete(todo)    # 위에서 찾아놓은 Todo.title == "우유 사기"인 todo 인스턴스가 session.deleted 목록에 추가
session.commit()        # 이 시점에서 DELETE SQL이 실행됨

print("삭제 후:", session.query(Todo).all())
# 삭제 후 4개 목록에서 3개 목록으로 줄어듬 (Todo.title == "우유 사기"가 삭제)

삭제 전: [<__main__.Todo object at 0x00000141B137AC30>, <__main__.Todo object at 0x00000141B2986210>, <__main__.Todo object at 0x00000141B2986490>, <__main__.Todo object at 0x00000141B1376C40>]
삭제 후: [<__main__.Todo object at 0x00000141B2986210>, <__main__.Todo object at 0x00000141B2986490>, <__main__.Todo object at 0x00000141B1376C40>]


In [19]:
# 데이터 준비

from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, String, Boolean, DateTime, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker

Base = declarative_base()

class Todo(Base):
    __tablename__ = "todos"
    id = Column(Integer, primary_key=True)
    title = Column(String, nullable=False)
    completed = Column(Boolean, default=False)
    priority = Column(Integer, default=1)
    category = Column(String)
    created_at = Column(DateTime, default=datetime.now)

engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(engine)

SessionLocal = sessionmaker(bind=engine)
session = SessionLocal()

todos = [
    Todo(title="우유 사기", completed=False, priority=2, category="장보기"),
    Todo(title="빵 사기", completed=True, priority=1, category="장보기"),
    Todo(title="보고서 작성", completed=False, priority=5, category="업무"),
    Todo(title="이메일 확인", completed=True, priority=3, category="업무"),
    Todo(title="운동하기", completed=False, priority=4, category="건강"),
    Todo(title="책 읽기", completed=False, priority=1, category="취미"),
    Todo(title="책 반납하기", completed=True, priority=2, category="기타"),
    Todo(title="회의 준비", completed=False, priority=5, category="업무"),
]
session.add_all(todos)
session.commit()

In [20]:
for todo in session.query(Todo).all():
    print(todo.id, todo.title, todo.completed, todo.priority, todo.category)

1 우유 사기 False 2 장보기
2 빵 사기 True 1 장보기
3 보고서 작성 False 5 업무
4 이메일 확인 True 3 업무
5 운동하기 False 4 건강
6 책 읽기 False 1 취미
7 책 반납하기 True 2 기타
8 회의 준비 False 5 업무


In [ ]:
# 1. 조건 필터링 (Filtering)

result1 = session.query(Todo).filter(Todo.id == 1).all()
# session.query(Todo) → 대상이 Todo 모델(테이블은 __tablename__ = "todos")인 조회 용도의 Query 객체 반환
# .filter() - 속성과 파이썬 비교 연산자 ==, !=, >, <, >=, <= 를 사용

print(result1)
# result1의 타입은 list[Todo객체]

print(".filter(Todo.id == 1):", [(t.id, t.title) for t in result1])

# 아래 셀에서 계속

.filter(Todo.id == 1): [(1, '우유 사기')]


In [ ]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기
# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 5 운동하기 False 4 건강
# 6 책 읽기 False 1 취미
# 7 책 반납하기 True 2 기타
# 8 회의 준비 False 5 업무


result2 = session.query(Todo).filter(Todo.category != "업무").all()
#         ---------ⓐ---------                                --ⓑ--

# ⓐ: session.query(Todo)의 의미는 조회 대상이 테이블 todos → SELECT ... from todos
# ⓑ: .all()의 리턴 타입은 Todo 인스턴스 리스트
# .filter() - 속성과 파이썬 비교 연산자 ==, !=, >, <, >=, <= 를 사용
print('.filter(Todo.category != "업무"):', [(t.id, t.category) for t in result2])


result3 = session.query(Todo).filter(Todo.priority > 3).all()
print(".filter(Todo.priority > 3):", [(t.id, t.priority) for t in result3])


result4 = session.query(Todo).filter(Todo.priority < 2).all()
print(".filter(Todo.priority < 2):", [(t.id, t.priority) for t in result4])


result5 = session.query(Todo).filter(Todo.priority >= 4).all()
print(".filter(Todo.priority >= 4):", [(t.id, t.priority) for t in result5])

# 아래 셀에서 계속

.filter(Todo.category != "업무"): [(1, '장보기'), (2, '장보기'), (5, '건강'), (6, '취미'), (7, '기타')]
.filter(Todo.priority > 3): [(3, 5), (5, 4), (8, 5)]
.filter(Todo.priority < 2): [(2, 1), (6, 1)]
.filter(Todo.priority >= 4): [(3, 5), (5, 4), (8, 5)]


In [ ]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기
# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 5 운동하기 False 4 건강
# 6 책 읽기 False 1 취미
# 7 책 반납하기 True 2 기타
# 8 회의 준비 False 5 업무

result6 = session.query(Todo).filter(Todo.priority <= 1).all()
print(".filter(Todo.priority <= 1):", [(t.id, t.priority) for t in result6])

# --------------------------------------------------------------------

# .filter_by() - "속성 이름=값"의 형태로 사용
# session.query(Todo).filter_by(id=1)
# session.query(Todo).filter_by(completed=True)
# session.query(Todo).filter_by(title="우유 사기")

result7 = session.query(Todo).filter_by(id=1).all()
print(".filter_by(id=1):", [(t.id, t.title) for t in result7])

result8 = session.query(Todo).filter_by(completed=True).all()
print(".filter_by(completed=True):", [(t.id, t.title) for t in result8])

.filter(Todo.priority <= 1): [(2, 1), (6, 1)]
.filter_by(id=1): [(1, '우유 사기')]
.filter_by(completed=True): [(2, '빵 사기'), (4, '이메일 확인'), (7, '책 반납하기')]


In [24]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기
# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 5 운동하기 False 4 건강
# 6 책 읽기 False 1 취미
# 7 책 반납하기 True 2 기타
# 8 회의 준비 False 5 업무

# 2. 논리 연산 조합
# .filter_by() - "속성 이름=값"의 형태로 사용이므로 해당 없음

from sqlalchemy import and_, or_, not_

# and_(■1, ■2, ■3,...) - 여러 조건을 모두 만족
result1 = session.query(Todo).filter(and_(Todo.category == "업무", Todo.completed == False)).all()
print("and_(category==업무, completed==False):", [(t.id, t.title) for t in result1])

# 아래 셀에서 계속

and_(category==업무, completed==False): [(3, '보고서 작성'), (8, '회의 준비')]


In [ ]:
# filter()에 콤마로 여러 조건을 나열해도 and_()와 동일
result2 = session.query(Todo).filter(Todo.category == "업무", Todo.completed == False).all()
print("filter(조건1, 조건2):", [(t.id, t.title) for t in result2])

# 아래 셀에서 계속

filter(조건1, 조건2): [(3, '보고서 작성'), (8, '회의 준비')]


In [ ]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기
# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 5 운동하기 False 4 건강
# 6 책 읽기 False 1 취미
# 7 책 반납하기 True 2 기타
# 8 회의 준비 False 5 업무

# or_() - 조건 중 하나만 만족
result3 = session.query(Todo).filter(or_(Todo.category == "건강", Todo.category == "취미")).all()
print("or_(category==건강, category==취미):", [(t.id, t.title) for t in result3])

# 아래 셀에서 계속

or_(category==건강, category==취미): [(5, '운동하기'), (6, '책 읽기')]


In [27]:
# not_() - negation
result4 = session.query(Todo).filter(not_(Todo.completed == True)).all()
print("not_(completed==True):", [(t.id, t.title) for t in result4])

# 아래 셀에서 계속

not_(completed==True): [(1, '우유 사기'), (3, '보고서 작성'), (5, '운동하기'), (6, '책 읽기'), (8, '회의 준비')]


In [ ]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기
# 3 보고서 작성 False 5 업무    <----
# 4 이메일 확인 True 3 업무     <----
# 5 운동하기 False 4 건강
# 6 책 읽기 False 1 취미
# 7 책 반납하기 True 2 기타
# 8 회의 준비 False 5 업무      <----

# and_() + or_() 조합 - '업무'이면서 (완료됐거나 우선순위가 5 이상)
result5 = session.query(Todo).filter(
    and_(
        Todo.category == "업무",
        or_(Todo.completed == True, Todo.priority >= 5)
    )
).all()
print("and_(업무, or_(완료, 우선순위>=5)):", [(t.id, t.title) for t in result5])

and_(업무, or_(완료, 우선순위>=5)): [(3, '보고서 작성'), (4, '이메일 확인'), (8, '회의 준비')]


In [ ]:
# [원본 데이터]                          [오름차순: priority]                    [내림차순: priority]
# 1 우유 사기 False 2 장보기              6 책 읽기 False 1 취미                  3 보고서 작성 False 5 업무
# 2 빵 사기 True 1 장보기                 2 빵 사기 True 1 장보기                 8 회의 준비 False 5 업무
# 3 보고서 작성 False 5 업무              1 우유 사기 False 2 장보기              5 운동하기 False 4 건강
# 4 이메일 확인 True 3 업무               7 책 반납하기 True 2 기타               4 이메일 확인 True 3 업무
# 5 운동하기 False 4 건강                 4 이메일 확인 True 3 업무               1 우유 사기 False 2 장보기
# 6 책 읽기 False 1 취미                  5 운동하기 False 4 건강                 7 책 반납하기 True 2 기타
# 7 책 반납하기 True 2 기타               3 보고서 작성 False 5 업무              6 책 읽기 False 1 취미
# 8 회의 준비 False 5 업무                8 회의 준비 False 5 업무                2 빵 사기 True 1 장보기

# 3. 정렬 (Ordering)

# .order_by(컬럼)의 형태로 사용하며 오름차순이 디폴트
result1 = session.query(Todo).order_by(Todo.priority).all()
# todos 테이블 (__tablename__ = "todos")의 priority 컬럼 기준으로 오름차순 정렬
print(".order_by(Todo.priority) [오름차순]:", [(t.id, t.priority) for t in result1])

# .order_by(컬럼.desc()) - 내림차순은 .desc()
result2 = session.query(Todo).order_by(Todo.priority.desc()).all()
# todos 테이블 (__tablename__ = "todos")의 priority 컬럼 기준으로 내림차순 정렬
print(".order_by(Todo.priority.desc()) [내림차순]:", [(t.id, t.priority) for t in result2])

# .order_by(컬럼1, 컬럼2) - 여러 컬럼 기준 정렬, 순서대로 적용
result3 = session.query(Todo).order_by(Todo.category, Todo.priority.desc()).all()
# 1차 기준은 Todo.category의 오름차순, 2차 기준은 Todo.priority의 내림차순
# 1차 기준 값이 서로 동일해서 순서를 정할 수 없는 경우에만 2차 기준이 추가로 적용
# -------------------→ 업무에 해당하며 priority는 5, 3, 5이므로 내림차순을 적용하면 5, 5, 3
print(".order_by(Todo.category, Todo.priority.desc()):", [(t.id, t.category, t.priority) for t in result3])

.order_by(Todo.priority) [오름차순]: [(2, 1), (6, 1), (1, 2), (7, 2), (4, 3), (5, 4), (3, 5), (8, 5)]
.order_by(Todo.priority.desc()) [내림차순]: [(3, 5), (8, 5), (5, 4), (4, 3), (1, 2), (7, 2), (2, 1), (6, 1)]
.order_by(Todo.category, Todo.priority.desc()): [(5, '건강', 4), (7, '기타', 2), (3, '업무', 5), (8, '업무', 5), (4, '업무', 3), (1, '장보기', 2), (2, '장보기', 1), (6, '취미', 1)]


In [ ]:
# 4. 개수 제한 / 페이지네이션 (Limit/Offset)

# .order_by(■).limit(N) - 상위 N개

# session.query(Todo).A().B().C().D().all()

result1 = session.query(Todo).order_by(Todo.id).limit(3).all()
# session.query(Todo).order_by(Todo.id).limit(3)까지는 아직 실행이 되지 않은 상태
# .all()을 실행해야 비로소 실행이 됨

print(".limit(3):", [t.id for t in result1])

# 아래 셀에서 계속

.limit(3): [1, 2, 3]


In [ ]:
# .order_by(■).offset(개수) - 앞에서 N개 건너뛰기

result2 = session.query(Todo).order_by(Todo.id).offset(5).all()
print(".offset(5):", [t.id for t in result2])

# 아래 셀에서 계속

.offset(5): [6, 7, 8]


In [30]:
# .limit() + .offset() 조합
result3 = session.query(Todo).order_by(Todo.id).offset(3).limit(2).all() # 앞에서 3개 건너뛰고 2개
print(".offset(3).limit(2)", [t.id for t in result3])

# 아래 셀에서 계속

.offset(3).limit(2) [4, 5]


In [31]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기
# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 5 운동하기 False 4 건강
# 6 책 읽기 False 1 취미
# 7 책 반납하기 True 2 기타
# 8 회의 준비 False 5 업무

# .limit() + .filter() 조합
result4 = session.query(Todo).filter(Todo.completed == False).order_by(Todo.id).limit(2).all()
print(".filter(completed=False).limit(2):", [t.id for t in result4])

.filter(completed=False).limit(2): [1, 3]


In [58]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기
# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 5 운동하기 False 4 건강
# 6 책 읽기 False 1 취미
# 7 책 반납하기 True 2 기타
# 8 회의 준비 False 5 업무

In [32]:
# .all()
result1 = session.query(Todo).filter(Todo.category == "업무").all()
print(".all():", [t.id for t in result1])

# 아래 셀에서 계속

.all(): [3, 4, 8]


In [ ]:
# .first() - 조회 결과 중 첫 번째 행 하나만
result2 = session.query(Todo).filter(Todo.category == "업무").order_by(Todo.id).first()
print(".first():", result2.id)

# 아래 셀에서 계속

.first(): 3


In [34]:
# .one() - 정확히 1건인 조건을 만족하면 리턴
result3 = session.query(Todo).filter(Todo.id == 1).one()
print(".one():", result3.id)

# 아래 셀에서 계속

.one(): 1


In [ ]:
# .one() - 2건 이상이면 MultipleResultsFound 예외 발생
try:
    session.query(Todo).filter(Todo.category == "업무").one()
except Exception as e:
    print(".one():", type(e).__name__)

# 아래 셀에서 계속

.one(): MultipleResultsFound


In [ ]:
# .one_or_none() - 없으면 None, 1개뿐이면 리턴, 2개 이상은 에러
#                  -------에러는 아님--------

# .one()          → 0개면 에러, 1개면 반환, 2개 이상이면 에러
# .one_or_none()  → 0개면 None, 1개면 반환, 2개 이상이면 에러
# .first()        → 0개면 None, 1개 이상이면 첫 번째만 반환

result4 = session.query(Todo).filter(Todo.id == 0).one_or_none()
print(".one_or_none():", result4)

# 아래 셀에서 계속

.one_or_none(): None


In [ ]:
# .scalar()

# 결과 행의 개수가   0개 → None
# 결과 행의 개수가   1개 → 첫 번째 컬럼 값 리턴
# 결과 행의 개수가 > 1개 → MultipleResultsFound 예외 발생

# 첫 번째 컬럼의 기준은 session.query(■)의 ■에 따라 달라짐


result5 = session.query(Todo.title).filter(Todo.id == 1).scalar()
#                 -----------------
#                 SELECT title → 보여줄 컬럼 (하나 밖에 없음)
#                                   --------------------
#                                    WHERE id=1 → 조건


# session.query(Todo)와 session.query(Todo.title)의 차이

# 둘의 공통점은 조회 대상이 Todo라는 점 (todos 테이블이 조회 대상)

# 둘의 차이점은 "무엇을 보여주는가?" → SELECT * vs. SELECT title
# session.query(Todo.title)는 보여줄 컬럼을 이야기 하는 것; title 컬럼을 SELECT

print(".scalar():", result5)
# 결과 행의 개수가 일단 1이고
# 보여줄 컬럼이 title 하나 밖에 없으므로 scalar()의 리턴은 title인 "우유 사기"

# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기
# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 5 운동하기 False 4 건강
# 6 책 읽기 False 1 취미
# 7 책 반납하기 True 2 기타
# 8 회의 준비 False 5 업무

.scalar(): 우유 사기


In [ ]:
from sqlalchemy import func

# 6. 집계 (Aggregation)

# .count() - 조건에 맞는 개수 세기
result1 = session.query(Todo).filter(Todo.category == "업무").count()
#         -------------------------------------------------- Query를 만드는 부분
# count()가 실행되어야 실제로 SQL문이 실행됨
print(".count() [카테고리가 업무인 개수]:", result1)

result2 = session.query(Todo).count()
print(".count() [전체 행 개수]:", result2)

# 아래 셀에서 계속

.count() [카테고리가 업무인 개수]: 3
.count() [전체 행 개수]: 8


In [ ]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기
# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 5 운동하기 False 4 건강
# 6 책 읽기 False 1 취미
# 7 책 반납하기 True 2 기타
# 8 회의 준비 False 5 업무

In [ ]:
# func.sum() - 합계
# 그냥 sum이라고 하면 파이썬의 sum과 이름이 충돌

result3 = session.query(func.sum(Todo.priority)).scalar()
# ⓐ func.sum(Todo.priority): todos 테이블의 priority 컬럼 값들을 전부 합산하라는 SQL 표현식
#    SELECT sum(priority) FROM todos은 SQL 문 
#    sum(priority)는 SQL 표현식

# ⓑ session.query(func.sum(Todo.priority)): 보여줄 대상은 func.sum(Todo.priority)
#    → "SELECT sum(todos.priority) FROM todos"와 같은 SQL 문이 만들어짐
# ⓒ .scalar(): 이 시점에 SQL이 실제로 데이터베이스에 전송/실행됨
# ⓓ func.sum(Todo.priority)를 보여주려면 결과 행의 개수가 1이 되고
#    보여줄 대상도 하나이므로 scalar()는 func.sum(Todo.priority)을 리턴

print("func.sum(priority) [전체 우선순위 합]:", result3)

# 아래 셀에서 계속

func.sum(priority) [전체 우선순위 합]: 23


In [ ]:
# func.avg(), func.max(), func.min() 모두 결과 행의 개수는 1
# 만약 group by를 적용하면 결과 행의 개수는 그룹 개수가 되어 1보다 클 수 있음

# func.avg() - 평균
result4 = session.query(func.avg(Todo.priority)).scalar()
print("func.avg(priority) [전체 우선순위 평균]:", result4)

# func.max(), func.min()
result5 = session.query(func.max(Todo.priority)).scalar()
print("func.max(priority):", result5)

result6 = session.query(func.min(Todo.priority)).scalar()
print("func.min(priority):", result6)

# 아래 셀에서 계속

func.avg(priority) [전체 우선순위 평균]: 2.875
func.max(priority): 5
func.min(priority): 1


In [ ]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기

# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 8 회의 준비 False 5 업무

# 5 운동하기 False 4 건강

# 6 책 읽기 False 1 취미

# 7 책 반납하기 True 2 기타


# .group_by() - 카테고리별 개수
result7 = session.query(Todo.category, func.count(Todo.id)).group_by(Todo.category).all()

# session.query(Todo.category, func.count(Todo.id)): 보여줄 대상은 Todo.category, func.count(Todo.id)
# "SELECT todos.category, count(todos.id) FROM todos" 라는 SQL이 만들어짐

# func.count(Todo.id)는 id 개수를 세는 SQL 표현식
# .group_by(Todo.category)
# category 값이 같은 행끼리 하나의 그룹으로 묶어서 func.count(Todo.id)을 계산

# .all(): 이 시점에 SQL이 실제로 데이터베이스에 전송/실행됨
# .all()의 리턴 타입은 리스트인데 ★★★★★★★★★★
# Todo 인스턴스의 리스트가 아니라, 튜플의 리스트가 되며 튜플의 구조는 (Todo.category, func.count(Todo.id))임

# group_by를 사용한다는 것은 그룹 개수별로 결과를 각각 산출하기 때문에 scalar()가 아닌 all()을 사용하는 것

print(".group_by(category) + count():", result7)

# 다음 셀에서 계속

# [■]
# ■? 튜플 (Todo.category, func.count(Todo.id))
# [(Todo.category, func.count(Todo.id)), (Todo.category, func.count(Todo.id)), (Todo.category, func.count(Todo.id))]

.group_by(category) + count(): [('건강', 1), ('기타', 1), ('업무', 3), ('장보기', 2), ('취미', 1)]


In [ ]:
# 1 우유 사기 False 2 장보기
# 2 빵 사기 True 1 장보기

# 3 보고서 작성 False 5 업무
# 4 이메일 확인 True 3 업무
# 8 회의 준비 False 5 업무

# 5 운동하기 False 4 건강

# 6 책 읽기 False 1 취미

# 7 책 반납하기 True 2 기타


# .group_by() + func.sum() - 카테고리별 우선순위 합
result8 = session.query(Todo.category, func.sum(Todo.priority)).group_by(Todo.category).all()
print(".group_by(category) + sum(priority):", result8)


.group_by(category) + sum(priority): [('건강', 4), ('기타', 2), ('업무', 13), ('장보기', 3), ('취미', 1)]


In [ ]:
# 7. 중복 제거

result1 = session.query(Todo.category).all()
# 이 경우 .all()의 타입은 튜플 리스트
# session.query()에 전달한 값이 모델 클래스(e.g., Todo)가 아니라 
# 속성 또는 표현식(e.g., Todo.category, func.count(...))이면 개수와 상관없이 결과는 튜플 리스트
# 예를 들어 session.query(■1, ■2, ■3).all()의 경우 [(■1, ■2, ■3), (■1, ■2, ■3), ...]

# 현재 리턴 타입은 [(Todo.category,)]의 형태
# 만약 튜플을 제거한다면
print("중복 제거 전 (전체 category):", [r for r in result1])
print("중복 제거 전 (전체 category):", [r[0] for r in result1])

# .distinct() - 중복된 category 값을 제거하고 유일한 값만
result2 = session.query(Todo.category).distinct().all() # session.query(Todo.category) 이니까 튜플
print(".distinct() [유일한 category만]:", [r[0] for r in result2]) # r[0]로 튜플 제거

중복 제거 전 (전체 category): [('장보기',), ('장보기',), ('업무',), ('업무',), ('건강',), ('취미',), ('기타',), ('업무',)]
중복 제거 전 (전체 category): ['장보기', '장보기', '업무', '업무', '건강', '취미', '기타', '업무']
.distinct() [유일한 category만]: ['장보기', '업무', '건강', '취미', '기타']


In [49]:
from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.orm import declarative_base, sessionmaker

Base = declarative_base()

class Todo(Base):
    __tablename__ = "todos"
    id = Column(Integer, primary_key=True)
    title = Column(String)
    completed = Column(Boolean)

engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(bind=engine)
session = SessionLocal()
session.add_all([
    Todo(title="우유 사기", completed=False),
    Todo(title="빵 사기", completed=True),
])
session.commit()

# session.query(모델)로 하면 결과는 list[모델 인스턴스]
result1 = session.query(Todo).all()
print("session.query(Todo).all()")
print("전체 결과:", result1)
print("    선택:", result1[0].title, result1[0].completed)

# session.query(컬럼1, 컬럼2)로 하면 결과는 list[(컬럼1, 컬럼2), (컬럼1, 컬럼2), ...)]
result2 = session.query(Todo.title, Todo.completed).all()
print("session.query(Todo.title, Todo.completed).all()")
print("전체 결과:", result2)
print("    선택:", result2[0][0], result2[0][1], "또는", result2[0].title, result2[0].completed) # 인덱스나 속성 이름으로

session.query(Todo).all()
전체 결과: [<__main__.Todo object at 0x00000141B2ACB390>, <__main__.Todo object at 0x00000141B1375940>]
    선택: 우유 사기 False
session.query(Todo.title, Todo.completed).all()
전체 결과: [('우유 사기', False), ('빵 사기', True)]
    선택: 우유 사기 False 또는 우유 사기 False
